# SIH26006 — Threshold-Based Classification Audit (A0 → A5)

**Isolation Contract**: This notebook is completely isolated from the production pipeline.
It does NOT modify any existing `outputs/` files or `src/` pipelines.
All outputs go to: `outputs/classification_audit/`

## Phases
- **A0** — Target landscape: 5 assets × 4 horizons × 4 thresholds = 80 class-distribution configs
- **A1** — 7 naive baselines (AlwaysUP/DOWN/NEUTRAL, MajorityTrain, PrevDirection, 7dTrend, MA5/20Cross)
- **A2** — Ridge (LogReg L2) classifier — features from TRAIN only, C tuned on VAL only, test evaluated ONCE
- **A3** — Leakage audit: verify label formula, no dir_* or target_* in features, random row check
- **A4** — Locked test comparison matrix
- **A5** — Verdict: does Ridge beat the strongest naive strategy?

**Label formula**: `pct_change(t) = (target_{asset}_{h}d[t] - asset[t]) / asset[t]`  
**Threshold selection**: based on TRAIN-set class balance only — NO test info used.

In [ ]:
# ================================================================================
# CELL 1: REPOSITORY SETUP
# ================================================================================
import os
import sys
import numpy as np
import pandas as pd

np.random.seed(42)

if not os.path.exists('outputs/modeling_dataset.csv'):
    if os.path.exists('FICOS-Platform/outputs/modeling_dataset.csv'):
        %cd FICOS-Platform
    else:
        !git clone https://github.com/SSOHEB/FICOS-Platform.git
        %cd FICOS-Platform

# Always pull the latest clean commits
!git pull origin main

os.makedirs('outputs/classification_audit', exist_ok=True)
print(f"Working directory : {os.getcwd()}")
print(f"Dataset exists    : {os.path.exists('outputs/modeling_dataset.csv')}")
print(f"Audit script exists: {os.path.exists('scratch/threshold_classification_audit.py')}")

In [ ]:
# ================================================================================
# CELL 2: RUN FULL AUDIT (A0 → A5)
# ================================================================================
# Estimated runtime: 5-12 min on Colab CPU (A2 Ridge fitting is the bottleneck)
# GPU is not needed; this is scikit-learn only
!python scratch/threshold_classification_audit.py

In [ ]:
# ================================================================================
# CELL 3: DISPLAY A0 — TARGET LANDSCAPE PIVOT
# ================================================================================
import pandas as pd

if os.path.exists('outputs/classification_audit/a0_target_landscape.csv'):
    df_a0 = pd.read_csv('outputs/classification_audit/a0_target_landscape.csv')
    # Show train-set distributions only
    a0_tr = df_a0[df_a0['split'] == 'train']
    print("=" * 70)
    print("A0 — TRAIN-SET CLASS DISTRIBUTIONS (flag: G=Good, ~=OK, N=TooNarrow, W=TooWide)")
    print("=" * 70)
    display(a0_tr[['asset','horizon','threshold_pct','UP%','DOWN%','NEUTRAL%',
                   'majority_class','majority_acc%','balance_flag']].to_string(index=False))
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 4: DISPLAY A1 — NAIVE BASELINE SUMMARY
# ================================================================================
if os.path.exists('outputs/classification_audit/a1_naive_baselines.csv'):
    df_a1 = pd.read_csv('outputs/classification_audit/a1_naive_baselines.csv')

    # Show best naive baseline per asset x horizon (all thresholds)
    print("=" * 70)
    print("A1 — BEST NAIVE BASELINE PER ASSET x HORIZON (all thresholds)")
    print("=" * 70)
    best_rows = (
        df_a1.sort_values('macro_F1%', ascending=False)
             .groupby(['asset','horizon','threshold_pct'], as_index=False)
             .first()
    )
    display(best_rows[['asset','horizon','threshold_pct','baseline',
                        'n_test','accuracy%','macro_F1%',
                        'F1_UP%','F1_DOWN%','F1_NEUTRAL%']])
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 5: DISPLAY A2 — RIDGE CLASSIFIER RESULTS
# ================================================================================
if os.path.exists('outputs/classification_audit/a2_ridge_classifier.csv'):
    df_a2 = pd.read_csv('outputs/classification_audit/a2_ridge_classifier.csv')
    print("=" * 70)
    print("A2 — RIDGE (LogReg L2) TEST SET RESULTS")
    print("=" * 70)
    display(df_a2[['asset','horizon','threshold_pct','best_C','n_features',
                   'val_macro_F1%','test_accuracy%','test_macro_F1%',
                   'test_F1_UP%','test_F1_DOWN%','test_F1_NEUTRAL%',
                   'top5_features']])
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 6: DISPLAY A3 — LEAKAGE AUDIT RESULT
# ================================================================================
if os.path.exists('outputs/classification_audit/a3_leakage_audit.csv'):
    df_a3 = pd.read_csv('outputs/classification_audit/a3_leakage_audit.csv')
    print("=" * 70)
    print("A3 — LEAKAGE AUDIT")
    print("=" * 70)
    display(df_a3)
    result = df_a3.iloc[0]['audit_result']
    if result == 'PASS':
        print("\n  VERDICT: PASS — no dir_* or target_* columns in feature matrix")
    else:
        print(f"\n  VERDICT: FAIL — leakage detected!")
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 7: DISPLAY A4+A5 — FINAL COMPARISON + VERDICT
# ================================================================================
if os.path.exists('outputs/classification_audit/a4_test_comparison.csv'):
    df_a4 = pd.read_csv('outputs/classification_audit/a4_test_comparison.csv')
    df_a5 = pd.read_csv('outputs/classification_audit/a5_verdict.csv')

    print("=" * 70)
    print("A4 — LOCKED TEST COMPARISON: Ridge vs Best Naive Baseline")
    print("=" * 70)
    df_a4_disp = df_a4.copy()
    df_a4_disp['ridge_wins'] = df_a4_disp['ridge_beats_naive'].map({True: 'RIDGE WINS', False: 'NAIVE BETTER'})
    df_a4_disp['margin'] = df_a4_disp['margin_F1'].apply(lambda x: f"{x:+.1f}%" if pd.notna(x) else 'N/A')
    display(df_a4_disp[['asset','horizon','rec_threshold','n_test',
                         'best_naive_baseline','best_naive_macF1%',
                         'ridge_macF1%','margin','ridge_wins']])

    print("\n" + "=" * 70)
    print("A5 — VERDICT")
    print("=" * 70)
    v = df_a5.iloc[0]
    print(f"  Ridge beats naive in : {int(v['beats_count'])}/{int(v['total_count'])} cases")
    print(f"  Avg winning margin   : +{v['avg_win_margin_F1']:.1f}% macro F1")
    print(f"  Avg losing margin    : {v['avg_lose_margin_F1']:.1f}% macro F1")
    print(f"  Best naive avg F1    : {v['best_naive_avg_macF1']:.1f}%")
    print(f"  Ridge avg F1         : {v['ridge_avg_macF1']:.1f}%")
    print(f"  Net Ridge edge       : {v['net_ridge_edge_F1']:+.1f}%")
    print(f"\n  CONCLUSION: {v['conclusion']}")
else:
    print('Run Cell 2 first.')